# 6.1 Traces

Trace output is the primary numerical result for most seismic simulations. FrequenSolve stores receiver data in HDF5 trace files and exposes them through `TraceDataset`, which reads metadata first and materializes arrays only when a frequency-domain or time-domain gather is requested. This tutorial runs a small acoustic model, inspects the trace store, plots frequency and time-domain data, and shows reload/export patterns.

By the end, you should be able to inspect trace metadata, read frequency-domain and reconstructed time-domain data, reopen trace files, and prepare export workflows.


## How To Read This Tutorial

Trace files are the bridge between numerical simulation and seismic analysis. FrequenSolve stores frequency-domain receiver responses with enough metadata to reconstruct time-domain gathers, identify sources/receivers/components, and reload results in a later notebook.

The story here is metadata first, arrays second: open the trace store, inspect what groups and components exist, read a frequency-domain gather, then reconstruct time-domain data with an explicit wavelet.

## Storage And Loading Model

| Layer | What it contains | User-facing API |
| --- | --- | --- |
| HDF5 trace files | Frequency-domain receiver data, source ids, receiver ids, components, and geometry tables | `result.traces()` or `fs.TraceDataset.open(...)` |
| `TraceDataset` | A lightweight facade around one or more trace files plus frequency metadata | `.summary`, `.groups`, `.components(group)`, `.sources(group)` |
| Frequency-domain read | Complex-valued `xarray.DataArray` indexed by frequency and receiver | `traces.fd(group, component, source)` |
| Time-domain read | Wavelet-reconstructed `xarray.DataArray` indexed by time and receiver | `traces.td(group, component, source, wavelet)` |
| SEG-Y export | Optional conversion of a time-domain gather | `gather.fs.to_segy(...)` with the `seismic-io` extra installed |

The API is intentionally based on `xarray`, so plotting, indexing, attributes, and coordinate-aware operations stay standard.


## Packed Trace Files And Lazy Reads

Solver runs may write per-frequency trace shards and then pack them into a single HDF5 trace product. Recent packed files store each frequency as a numbered dataset under `/trace_data` with an index table under `/trace_index`; older packed files may expose a direct frequency axis. `TraceDataset` handles both layouts and presents the same `xarray` interface.

This matters operationally because frequencies can be inserted or packed out of order while the Python reader still resolves the physical frequency coordinate from metadata. Use `traces.summary`, `traces.frequencies(group)`, and `traces.fd(...)` rather than assuming filenames or dataset numbers imply frequency order.

## Imports

The examples use the public `import frequensolve as fs` API plus standard scientific Python tools for inspection and plotting. Keeping imports ordinary makes the notebook easier to reuse in analysis or operations notebooks.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import frequensolve as fs

u = fs.ureg


## Build A Trace-Producing Acoustic Job

Trace output is enabled by default for jobs. The explicit `TraceOutput` class is useful when you want to change the trace directory, but most tutorials can use the default and access traces through the completed result.


In [ ]:
project = fs.Project(
    name="project",
    pretty_name="trace_outputs",
    path="./scratch/tutorials/traces",
    log_level="INFO",
    log_to_console=True,
)

sim = project.new_simulation(
    name="trace_outputs",
    physics="acoustic",
    dimension=2,
    units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
)

model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.0])
model.add_surface(name="top", depth=0.0 * u.km)
model.add_layer(name="water", properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3})
model.add_surface(name="interface", depth=0.22 * u.km)
model.add_layer(name="basement", properties={"Vp": 2.4 * u.km / u.s, "Rho": 2.2 * u.g / u.cm**3})
model.add_surface(name="bottom", depth=0.5 * u.km)
sim += model

sim += model.hex_mesh_generator([8, 4])
sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=30.0)
sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(conditions=["pml"], boundaries=["x_min", "x_max", "z_max"], pml_wavelengths=0.75)

acq = fs.Acquisition()
acq.add_source_group(kind="scalar", coords=[[0.35, 0.05], [0.65, 0.05]])
hydrophone = fs.ReceiverNode(name="hydrophone")
hydrophone.add_component(name="p", field="pressure")
receiver_coords = [[x, 0.04] for x in np.linspace(0.1, 0.9, 81)]
acq.add_receiver_group(name="surface", device=hydrophone, coords=receiver_coords)
sim += acq
sim += fs.Discretization()
sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)

model.plot("vp", figsize=(7, 3), aspect="equal")


## Run And Open The Trace Dataset

The run cell is strict. After it succeeds, `result.traces(upscale=...)` returns a `TraceDataset`. Upscaling controls the time-domain reconstruction sampling; it does not rerun the solver.


In [ ]:
site = fs.Site()
job = fs.TimeDomainJob(
    name="time_traces",
    simulation=sim,
    f_min=0.0,
    f_max=30.0,
    T_max=0.9,
)
result = site.submit(job).wait()
traces = result.traces(upscale=4)
traces.summary


## Inspect Metadata Before Reading Data

The metadata and summary calls are cheap compared with loading arrays. Use them to choose the receiver group, component, source id, and frequency range for later reads.


In [ ]:
{
    "trace_files": traces.files,
    "groups": traces.groups,
    "components": {group: traces.components(group) for group in traces.groups},
    "sources": {group: traces.sources(group) for group in traces.groups},
    "frequencies_hz": traces.frequencies().tolist(),
}


## Read Frequency-Domain Data

Frequency-domain reads return complex-valued `xarray.DataArray` objects. The example below plots the average pressure spectrum over receivers for each source. This is a useful first check for frequency coverage and source consistency.


In [ ]:
group = "surface"
component = "p"
source_ids = traces.sources(group)
fd_by_source = {
    source_id: traces.fd(group, component, source=source_id)
    for source_id in source_ids
}

fig, ax = plt.subplots(figsize=(8, 4))
for source_id, fd in fd_by_source.items():
    spectrum = np.abs(fd).mean(dim="receiver")
    ax.plot(spectrum["frequency"].values, spectrum.values, marker="o", label=f"source {source_id}")
ax.set_xlabel("Frequency")
ax.set_ylabel("Mean |pressure|")
ax.set_title("Frequency-domain receiver spectrum")
ax.legend()
fig.tight_layout()


## Reconstruct And Plot Time-Domain Gathers

A time-domain gather combines the computed frequency response with a source wavelet. Changing the wavelet changes the reconstruction, not the underlying solver result.


`RickerWavelet(f=12.0)` uses one period of center padding by default, so the wavelet peak is aligned at physical time zero. Pass `center=` only when you intentionally want a different amount of negative-time padding.


In [ ]:
wavelet = fs.RickerWavelet(f=12.0)
td_by_source = {
    source_id: traces.td(group, component, source_id, wavelet, upscale=4, T_max=0.9)
    for source_id in source_ids
}

A = 2.0 * max(float(np.nanstd(np.real(gather.values))) for gather in td_by_source.values())
fig, axes = plt.subplots(1, len(td_by_source), figsize=(5 * len(td_by_source), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (source_id, gather) in zip(axes, td_by_source.items()):
    fs.plot_gather(gather, ax=ax, A=A, cmap="gray", title=f"source {source_id}")
fig.tight_layout()


## Reload From Disk And Consolidate

`TraceDataset.open(...)` can reopen an existing trace file outside the original run handle. This is the workflow to use in analysis notebooks, batch post-processing, or after fetching results from a remote site.

`consolidate()` builds or refreshes the virtual dataset cache used for faster repeated reads. It should not change the physical trace values; it organizes trace shards and metadata so future reads can select by group, component, source, and frequency without walking every raw output file each time.


In [ ]:
first_trace_file = traces.paths[0]
reopened = fs.TraceDataset.open(first_trace_file, upscale=4)
cache_file = traces.consolidate()
{
    "opened_file": str(first_trace_file),
    "reopened_groups": reopened.groups,
    "cache_file": str(cache_file),
}


## Optional SEG-Y Export

SEG-Y export is available from the returned `xarray.DataArray` via the `.fs` accessor. It requires the optional `seismic-io` dependency set because the writer uses `segyio`.


In [ ]:
# Uncomment after installing the seismic I/O extra:
# segy_path = Path("./assets/trace_outputs_source1.sgy")
# segy_path.parent.mkdir(exist_ok=True)
# td_by_source[source_ids[0]].fs.to_segy(segy_path, units_in="km", units_out="m")
# segy_path


## Before Moving On

A trace plot is only meaningful after the metadata is understood. Always check receiver group names, component names, source ids, and frequency coverage before selecting a gather.

For production analysis, prefer loading saved trace datasets from disk or fetched result bundles rather than relying on the original run handle. That makes post-processing reproducible and independent of the execution notebook.

## Result Review Checklist

Trace output should be reviewed at the metadata level before plotting. That keeps source, receiver, and component selection explicit and avoids confusing a plotting choice with a data-layout issue.

| Artifact | What to confirm |
| --- | --- |
| `traces.summary` | Receiver groups, components, source ids, and frequency coverage match the job. |
| `traces.fd(...)` | Complex frequency-domain data are present before wavelet reconstruction. |
| `traces.td(...)` | The chosen wavelet and `T_max` produce the expected time window. |
| Reopened dataset | `TraceDataset.open(...)` can load the trace file outside the original result handle. |
| SEG-Y export path | Optional export is performed from an `xarray.DataArray` after units are chosen. |

When a gather looks empty, first inspect frequency coverage, source ids, and component names. Those checks are cheaper and more reliable than changing wavelets or plotting scales.
